# Salud Chilecito - Demo DAO y plataforma

Este notebook funciona como recorrido de presentacion de Salud Chilecito. Primero se valida el modelo de datos de demo, despues se revisa el DAO y finalmente se explican los comandos para conectar con Oracle.

**Formas de uso del proyecto:**

- `http://localhost:8000`: plataforma grafica.
- `src/dao`: capa DAO para Oracle.
- `sql/`: scripts Oracle con tablespaces, usuarios, esquema, indices, seed y validaciones.

## 1. Preparacion

Ejecutar el notebook desde la raiz del repositorio:

```bash
Windows: .\scripts\windows\04_abrir_notebook.ps1
Ubuntu: bash scripts/ubuntu/04_abrir_notebook.sh
```

Si todavia no se instalaron dependencias:

```bash
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt
```


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print("Repositorio:", ROOT)
print("Existe src/dao:", (ROOT / "src" / "dao").exists())
print("Existe sql/:", (ROOT / "sql").exists())


## 2. Datos de demo

La plataforma web puede funcionar aunque Oracle no este levantado. Para eso usa `data/demo_seed.json` y guarda cambios de prueba en `runtime/salud_chilecito_data.json`.


In [ ]:
seed_path = ROOT / "data" / "demo_seed.json"
data = json.loads(seed_path.read_text(encoding="utf-8"))

for name in ["centros", "especialidades", "medicos", "pacientes", "turnos", "agendas", "tarifas", "documentos"]:
    print(f"{name}: {len(data[name])}")


In [ ]:
import pandas as pd

pd.DataFrame(data["centros"])


## 3. Uso del store local

Esta parte demuestra operaciones reales sin tocar Oracle. Sirve para probar la interfaz y validar la funcionalidad sin preparar una base externa.


In [ ]:
from tempfile import TemporaryDirectory
from src.webapp.store import JsonStore

tmp = TemporaryDirectory()
demo_store = JsonStore(
    runtime_path=Path(tmp.name) / "runtime.json",
    seed_path=seed_path,
    uploads_dir=Path(tmp.name) / "uploads",
)

# Crear paciente con centro_id (requerido para modelo single-hospital)
paciente = demo_store.create_patient({
    "dni": "50999888",
    "nombre": "Paciente Notebook",
    "telefono": "3825-111111",
    "distrito": "Chilecito",
    "obra_social": "APOS",
    "centro_id": 1,  # Hospital Eleazar Herrera Motta
})

turno = demo_store.create_turno({
    "paciente_id": paciente["id"],
    "medico_id": 1,
    "fecha": "2026-06-20",
    "hora": "09:30",
    "estado": "CONFIRMADO",
    "precio": 0,
    "motivo": "Control desde notebook",
})

print("Paciente creado:", paciente)
print("Turno creado:", turno["id"], turno["paciente"]["nombre"], turno["medico"]["nombre"])

## 4. Capa DAO para Oracle

In [ ]:
Los DAO son la interfaz formal entre Python y Oracle. La aplicación no debería armar consultas dispersas por todos lados; las consultas quedan agrupadas en clases como `CentroDAO`, `PacienteDAO` y `TurnoDAO`.

## 5. Ejemplo de uso con SQLAlchemy (Patrón del profesor)

In [ ]:
# Ejemplo de uso con SQLAlchemy (similar al patrón del profesor)
# El profesor usa: from sqlalchemy import create_engine, select, join, MetaData, Table
# Y usa pandas para visualizar los resultados

if engine:
    with engine.connect() as conn:
        # Ejemplo 1: Listar especialidades (similar a get_medical_specialities del profesor)
        especialidades = pd.read_sql(text("SELECT * FROM especialidades"), conn)
        print("Especialidades:")
        print(especialidades)
        
        # Ejemplo 2: Listar médicos con sus especialidades (JOIN)
        medicos = pd.read_sql(text("""
            SELECT m.id, m.nombre as medico, e.nombre as especialidad 
            FROM medicos m 
            JOIN especialidades e ON m.especialidad_id = e.id
        """), conn)
        print("\nMédicos con especialidades:")
        print(medicos)
else:
    print("Modo demo: Mostrando datos de demo_seed.json")
    print("\nEspecialidades:")
    print(pd.DataFrame(data["especialidades"]))
    print("\nMédicos:")
    print(pd.DataFrame(data["medicos"]))

In [ ]:
## 4. Capa DAO para Oracle

Los DAO son la interfaz formal entre Python y Oracle. La aplicación no debería armar consultas dispersas por todos lados; las consultas quedan agrupadas en clases como `CentroDAO`, `PacienteDAO` y `TurnoDAO`.

**Patrón del profesor (hdrobins/dao):**
- Usa SQLAlchemy con `create_engine`, `select`, `join`, `MetaData`, `Table`
- Modelos separados en `db_models/`
- Clase DAO principal con métodos específicos
- `config_vars.py` para conexión a base de datos

In [ ]:
# Importar DAOs del proyecto (siguiendo patrón similar al del profesor)
try:
    from src.dao import CentroDAO, PacienteDAO, TurnoDAO, MedicoDAO, EspecialidadDAO
    print("DAO disponibles:")
    print("-", CentroDAO.__name__)
    print("-", PacienteDAO.__name__)
    print("-", TurnoDAO.__name__)
    print("-", MedicoDAO.__name__)
    print("-", EspecialidadDAO.__name__)
except Exception as e:
    print(f"Error al importar DAOs: {e}")

# Demo del patrón DAO similar al del profesor
# Si hay conexión a Oracle, usamos SQLAlchemy directamente
if engine:
    with engine.connect() as conn:
        # Ejemplo: listar centros (similar al patrón del profesor)
        dataset = pd.read_sql(text("SELECT * FROM centros"), conn)
        print(dataset)
else:
    # Modo demo: usar datos JSON
    print("Modo demo: Mostrando datos de demo_seed.json")
    pd.DataFrame(data["centros"])

# Demo del patrón DAO similar al del profesor
# Si hay conexión a Oracle, usamos SQLAlchemy directamente
if engine:
    with engine.connect() as conn:
        # Ejemplo: listar centros (similar al patrón del profesor)
        dataset = pd.read_sql(text("SELECT * FROM centros"), conn)
        print("Centros:")
        print(dataset)
        
        # Ejemplo: listar médicos
        dataset_medicos = pd.read_sql(text("SELECT * FROM medicos"), conn)
        print("\nMédicos:")
        print(dataset_medicos)
else:
    # Modo demo: usar datos JSON
    print("Modo demo: Mostrando datos de demo_seed.json")
    print("\nCentros:")
    print(pd.DataFrame(data["centros"]))
    print("\nMédicos:")
    print(pd.DataFrame(data["medicos"]))

## 7. Resumen para el Profesor

**Modelo de negocio**: Sistema vendido a hospitales/clínicas (single-hospital instance)

**Autores**: Alesandro David Fajardo / Kevin Facundo Nunez  
**Universidad**: Universidad Nacional de Chilecito  
**Año**: 2026

### Características principales de la plataforma

1. **Gestión completa**: Centros, médicos, pacientes, turnos, agendas, documentos
2. **Selección por síntomas**: Derivación automática a especialidades (dolor de pecho → Cardiología)
3. **Precios personalizados**: Por especialidad y tipo de consulta (General, Urgencia, Seguimiento)
4. **Configuración por hospital**: Branding personalizado (logo, colores, mensajes de bienvenida)
5. **Disponibilidad en tiempo real**: Días y horarios específicos para los próximos 7 días
6. **Integración con HIS**: Se conecta a sistemas existentes, no los reemplaza
7. **API REST estándar**: Para integración con otros sistemas

### Arquitectura técnica

- **Base de datos**: Oracle con scripts completos (tablespaces, usuarios, esquema, índices, seed)
- **Capa DAO**: Python con patrón Data Access Object para abstracción de base de datos
- **API REST**: Endpoints estándar con documentación OpenAPI
- **Frontend**: JavaScript con visualización mejorada de disponibilidad

### Ventajas competitivas

1. **No reemplaza**: Los hospitales mantienen sus sistemas existentes
2. **Fácil integración**: API estándar y documentación clara
3. **Valor agregado**: Funcionalidades que los HIS no tienen (IA, selección por síntomas)
4. **Flexibilidad**: Se adapta a diferentes sistemas mediante adaptadores
5. **Costo efectivo**: Menor costo que migrar a un sistema completo

### Comandos para ejecutar

```bash
# Instalar dependencias
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # Ubuntu
pip install -r requirements.txt

# Ejecutar servidor
python -m src.webapp.server
```

### URLs de acceso

- `http://localhost:8000` - Plataforma gráfica principal
- `http://localhost:8000/landing` - Landing page del hospital

### Documentación completa

- [README](../README.md) - Documentación general del proyecto
- [Arquitectura de Integración](../docs/ARQUITECTURA_INTEGRACION.md) - Guía de integración con HIS
- [Documentación de API](../docs/API_OPENAPI.md) - Referencia completa de la API REST
- [Ejemplos de Integración](../examples/integracion_his.py) - Ejemplos de código completos
- [Integración con Hospital](../docs/INTEGRACION_HOSPITAL.md) - Guía de integración específica

## 7. Comandos finales de presentacion

```bash
python scripts/check_requirements.py
python -m pytest -q
python -m src.webapp.server
```

Abrir:

- `http://localhost:8000`